# Lab | Data Aggregation and Filtering

In this challenge, we will continue to work with customer data from an insurance company. We will use the dataset called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by first performing data cleaning, formatting, and structuring.

1. Create a new DataFrame that only includes customers who:
   - have a **low total_claim_amount** (e.g., below $1,000),
   - have a response "Yes" to the last marketing campaign.

In [2]:
import pandas as pd

# Load the dataset
url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv"

df = pd.read_csv(url)

df.shape

(10910, 26)

In [3]:
df.head()

,Unnamed: 0,Customer,State,Customer Lifetime Value,Response,Coverage,Education,Effective To Date,EmploymentStatus,Gender,...,Number of Open Complaints,Number of Policies,Policy Type,Policy,Renew Offer Type,Sales Channel,Total Claim Amount,Vehicle Class,Vehicle Size,Vehicle Type
0,0,DK49336,Arizona,4809.216960,No,Basic,College,2/18/11,Employed,M,...,0.0,9,Corporate Auto,Corporate L3,Offer3,Agent,292.800000,Four-Door Car,Medsize,NaN
1,1,KX64629,California,2228.525238,No,Basic,College,1/18/11,Unemployed,F,...,0.0,1,Personal Auto,Personal L3,Offer4,Call Center,744.924331,Four-Door Car,Medsize,NaN
2,2,LZ68649,Washington,14947.917300,No,Basic,Bachelor,2/10/11,Employed,M,...,0.0,2,Personal Auto,Personal L3,Offer3,Call Center,480.000000,SUV,Medsize,A
3,3,XL78013,Oregon,22332.439460,Yes,Extended,College,1/11/11,Employed,M,...,0.0,2,Corporate Auto,Corporate L3,Offer2,Branch,484.013411,Four-Door Car,Medsize,A
4,4,QA50777,Oregon,9025.067525,No,Premium,Bachelor,1/17/11,Medical Leave,F,...,NaN,7,Personal Auto,Personal L2,Offer1,Branch,707.925645,Four-Door Car,Medsize,NaN


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10910 entries, 0 to 10909
Data columns (total 26 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Unnamed: 0                     10910 non-null  int64  
 1   Customer                       10910 non-null  str    
 2   State                          10279 non-null  str    
 3   Customer Lifetime Value        10910 non-null  float64
 4   Response                       10279 non-null  str    
 5   Coverage                       10910 non-null  str    
 6   Education                      10910 non-null  str    
 7   Effective To Date              10910 non-null  str    
 8   EmploymentStatus               10910 non-null  str    
 9   Gender                         10910 non-null  str    
 10  Income                         10910 non-null  int64  
 11  Location Code                  10910 non-null  str    
 12  Marital Status                 10910 non-null  str    
 1

CLEANING

In [5]:
# Standardize column names
df.columns = df.columns.str.lower().str.replace(" ", "_")

df.columns

Index(['unnamed:_0', 'customer', 'state', 'customer_lifetime_value',
       'response', 'coverage', 'education', 'effective_to_date',
       'employmentstatus', 'gender', 'income', 'location_code',
       'marital_status', 'monthly_premium_auto', 'months_since_last_claim',
       'months_since_policy_inception', 'number_of_open_complaints',
       'number_of_policies', 'policy_type', 'policy', 'renew_offer_type',
       'sales_channel', 'total_claim_amount', 'vehicle_class', 'vehicle_size',
       'vehicle_type'],
      dtype='str')

In [6]:
# Check for duplicate rows
df.duplicated().sum()

np.int64(0)

In [7]:
# Inspect the complaint format
df["number_of_open_complaints"].head(10)

0    0.0
1    0.0
2    0.0
3    0.0
4    NaN
5    0.0
6    3.0
7    0.0
8    0.0
9    0.0
Name: number_of_open_complaints, dtype: float64

In [8]:
df["effective_to_date"].head()

0    2/18/11
1    1/18/11
2    2/10/11
3    1/11/11
4    1/17/11
Name: effective_to_date, dtype: str

In [15]:
# Convert effective_to_date to datetime format
df["effective_to_date"] = pd.to_datetime(
    df["effective_to_date"],
    format="%m/%d/%y"
)

In [16]:
# Filter customers with low claims and a positive response
low_risk_customers = df[
    (df["total_claim_amount"] < 1000)
    & (df["response"] == "Yes")
]

low_risk_customers.shape

(1399, 25)

2. Using the original Dataframe, analyze:
   - the average `monthly_premium` and/or customer lifetime value by `policy_type` and `gender` for customers who responded "Yes", and
   - compare these insights to `total_claim_amount` patterns, and discuss which segments appear most profitable or low-risk for the company.

In [17]:
# Filter customers who responded positively to the campaign
yes_customers = df[df["response"] == "Yes"]

yes_customers.shape

(1466, 25)

In [18]:
# Calculate average metrics by policy type and gender
profitability_summary = yes_customers.groupby(
    ["policy_type", "gender"]
)[
    ["monthly_premium_auto",
     "customer_lifetime_value",
     "total_claim_amount"]
].mean().round(2)

profitability_summary

monthly_premium_auto  customer_lifetime_value  \
policy_type    gender                                                  
Corporate Auto F                      94.30                  7712.63   
               M                      92.19                  7944.47   
Personal Auto  F                      99.00                  8339.79   
               M                      91.09                  7448.38   
Special Auto   F                      92.31                  7691.58   
               M                      86.34                  8247.09   

                       total_claim_amount  
policy_type    gender                      
Corporate Auto F                   433.74  
               M                   408.58  
Personal Auto  F                   452.97  
               M                   457.01  
Special Auto   F                   453.28  
               M                   429.53

Analysis:
Female customers with Personal Auto policies have the highest average customer lifetime value (8339.79) and the highest monthly premium (99.00), suggesting that this segment is highly profitable.

Male customers with Corporate Auto policies appear to be a lower-risk segment, as they combine a relatively high customer lifetime value (7944.47) with the lowest average total claim amount (408.58).

Overall, customers with lower claim amounts and higher lifetime value may represent the most profitable segments for the company.

3. Analyze the total number of customers who have policies in each state, and then filter the results to only include states where there are more than 500 customers.

In [19]:
# Count the number of customers by state
customers_by_state = df.groupby("state")["customer"].count()

customers_by_state

state
Arizona       1937
California    3552
Nevada         993
Oregon        2909
Washington     888
Name: customer, dtype: int64

In [20]:
# Filter states with more than 500 customers
large_states = customers_by_state[customers_by_state > 500]

large_states

state
Arizona       1937
California    3552
Nevada         993
Oregon        2909
Washington     888
Name: customer, dtype: int64

Analysis:
California has the largest customer base with 3,552 customers, followed by Oregon and Arizona. Washington and Nevada have fewer customers but still exceed the 500-customer threshold. These results suggest that California and Oregon are the company's strongest markets and may represent important regions for future marketing campaigns and customer retention efforts.

4. Find the maximum, minimum, and median customer lifetime value by education level and gender. Write your conclusions.

In [21]:
# Calculate max, min, and median CLV by education level and gender
clv_summary = df.groupby(
    ["education", "gender"]
)["customer_lifetime_value"].agg(
    ["max", "min", "median"]
).round(2)

clv_summary

max      min   median
education            gender                            
Bachelor             F       73225.96  1904.00  5640.51
                     M       67907.27  1898.01  5548.03
College              F       61850.19  1898.68  5623.61
                     M       61134.68  1918.12  6005.85
Doctor               F       44856.11  2395.57  5332.46
                     M       32677.34  2267.60  5577.67
High School or Below F       55277.45  2144.92  6039.55
                     M       83325.38  1940.98  6286.73
Master               F       51016.07  2417.78  5729.86
                     M       50568.26  2272.31  5579.10

In [23]:
# Calculate max, min, and median CLV by education level and gender
clv_summary = df.groupby(
    ["education", "gender"]
)["customer_lifetime_value"].agg(
    ["max", "min", "median"]
).round(2)

clv_summary

max      min   median
education            gender                            
Bachelor             F       73225.96  1904.00  5640.51
                     M       67907.27  1898.01  5548.03
College              F       61850.19  1898.68  5623.61
                     M       61134.68  1918.12  6005.85
Doctor               F       44856.11  2395.57  5332.46
                     M       32677.34  2267.60  5577.67
High School or Below F       55277.45  2144.92  6039.55
                     M       83325.38  1940.98  6286.73
Master               F       51016.07  2417.78  5729.86
                     M       50568.26  2272.31  5579.10

Analysis:
Customer lifetime value varies across education levels and gender. Male customers with a High School or Below education have the highest maximum CLV (83,325.38) and the highest median CLV (6,286.73). In contrast, customers with a Doctor degree show lower maximum values, especially among males.

Overall, the median values are much closer across groups than the maximum values, suggesting that a few high-value customers strongly influence some segments. Education level alone does not appear to be a strong predictor of customer lifetime value.

## Bonus

5. The marketing team wants to analyze the number of policies sold by state and month. Present the data in a table where the months are arranged as columns and the states are arranged as rows.

In [24]:
# Extract the month from the date
df["month"] = df["effective_to_date"].dt.month

In [25]:
# Count the number of policies sold by state and month
policies_by_state_month = df.pivot_table(
    values="policy",
    index="state",
    columns="month",
    aggfunc="count"
)

policies_by_state_month

month,1,2
state,,
Arizona,1008,929
California,1918,1634
Nevada,551,442
Oregon,1565,1344
Washington,463,425


Analysis:
California sold the highest number of policies in both months, followed by Oregon and Arizona. All states experienced a decrease in policy sales from January to February. California and Oregon appear to be the company's strongest markets and may deserve additional marketing investment.

6.  Display a new DataFrame that contains the number of policies sold by month, by state, for the top 3 states with the highest number of policies sold.

*Hint:*
- *To accomplish this, you will first need to group the data by state and month, then count the number of policies sold for each group. Afterwards, you will need to sort the data by the count of policies sold in descending order.*
- *Next, you will select the top 3 states with the highest number of policies sold.*
- *Finally, you will create a new DataFrame that contains the number of policies sold by month for each of the top 3 states.*

In [26]:
# Count the total number of policies by state
top_states = df["state"].value_counts().head(3)

top_states

state
California    3552
Oregon        2909
Arizona       1937
Name: count, dtype: int64

In [27]:
# Keep only the top 3 states
top_3_policies = policies_by_state_month.loc[top_states.index]

top_3_policies

month,1,2
state,,
California,1918,1634
Oregon,1565,1344
Arizona,1008,929


In [28]:
# Keep only the top 3 states
top_3_policies = policies_by_state_month.loc[top_states.index]

top_3_policies

month,1,2
state,,
California,1918,1634
Oregon,1565,1344
Arizona,1008,929


7. The marketing team wants to analyze the effect of different marketing channels on the customer response rate.

Hint: You can use melt to unpivot the data and create a table that shows the customer response rate (those who responded "Yes") by marketing channel.

External Resources for Data Filtering: https://towardsdatascience.com/filtering-data-frames-in-pandas-b570b1f834b9

In [29]:
# Calculate the response rate by sales channel
response_rate = df.groupby("sales_channel")["response"].apply(
    lambda x: (x == "Yes").mean() * 100
).round(2)

response_rate

sales_channel
Agent          18.01
Branch         10.79
Call Center    10.32
Web            10.89
Name: response, dtype: float64

In [30]:
# Convert the Series into a DataFrame
response_rate_df = response_rate.reset_index()

response_rate_df.columns = [
    "sales_channel",
    "response_rate"
]

response_rate_df

,sales_channel,response_rate
0,Agent,18.01
1,Branch,10.79
2,Call Center,10.32
3,Web,10.89


In [31]:
# Convert the table into a long format
response_rate_long = response_rate_df.melt(
    id_vars="sales_channel",
    var_name="metric",
    value_name="value"
)

response_rate_long

,sales_channel,metric,value
0,Agent,response_rate,18.01
1,Branch,response_rate,10.79
2,Call Center,response_rate,10.32
3,Web,response_rate,10.89


Analysis:
The Agent channel achieved the highest response rate (18.01%), significantly outperforming the other marketing channels. The Web, Branch, and Call Center channels produced similar response rates, all around 10–11%. These results suggest that direct interactions with agents are more effective at engaging customers and encouraging positive responses to marketing campaigns.